Last Updated: May 2026
(Tested on Google Colab)

In [ ]:
!python --version

Python 3.12.13


# IMPORTANT

1. Run the installation cell first
2. Add required API keys in Colab Secrets
3. Run notebook cells sequentially

# If running in Google Colab Please ensure you update below keys in "secrets" on the left and give access to this notebook

1.   OPENAI_API_KEY
2.   TAVILY_API_KEY

In [ ]:
!pip install -q \
langchain==0.3.14 \
langchain-openai==0.2.14 \
langchain-community==0.3.14 \
openai==1.59.6 \
python-dotenv==1.0.1 \
requests==2.32.3 \
beautifulsoup4==4.12.3 \
wikipedia==1.4.0 \
ipykernel==6.29.5 \
tavily-python==0.5.0


In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
#Langchain
from langchain.tools import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.agents import initialize_agent, AgentType
from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
#If executing from local machine, run below 2 lines to load keys (.env should be present in same directory with keys in it)
# from dotenv import load_dotenv
# load_dotenv()


#If executing from Colab, run below lines to load keys (keys should be added in colab secrets and access given to this notebook)
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#prompt templates
prompt_template = PromptTemplate(
    input_variables=["name"],
    template="Hello, {name}! How can I help you today?"
)

formatted_prompt = prompt_template.format(name="David")
print(formatted_prompt)

Hello, David! How can I help you today?


In [ ]:
chat_model = ChatOpenAI(model="gpt-4o-mini")

response = chat_model.invoke("What is Capital of USA?")
print(response.content)

The capital of the United States is Washington, D.C.


In [ ]:
#LLM Chains
llm = ChatOpenAI(model="gpt-4o-mini")

learn_template = """
I want you to act as a consultant for a AI training
Return a list of topics and why it is important to learn in given area of AI
The description should be relevant to recent advancement in AI
What are some good topics to learn in {AI_topic}
"""

learn_prompt = PromptTemplate(
    input_variables=["AI_topic"],
    template=learn_template,
)

description = "Deep learning"

chain = LLMChain(llm=llm, prompt=learn_prompt)

result = chain.invoke({"AI_topic": description})
print(result["text"])

Deep learning is a rapidly evolving field within artificial intelligence (AI) that has seen significant advancements in recent years. Below is a list of key topics in deep learning, along with their importance in the context of recent developments:

### 1. **Neural Network Architectures**
   - **Importance**: Understanding different architectures like Convolutional Neural Networks (CNNs), Recurrent Neural Networks (RNNs), and Transformer networks is crucial as they serve as the backbone for various applications in computer vision, natural language processing, and more. Recent advancements have led to more efficient and powerful architectures, such as Vision Transformers (ViTs) and novel variations of RNNs.

### 2. **Transfer Learning and Fine-Tuning**
   - **Importance**: Transfer learning allows practitioners to leverage pre-trained models on large datasets and adapt them to specific tasks, significantly reducing the resources required for training. This is especially relevant with ad

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool

# Define a simple custom tool
def my_tool_function(query: str) -> str:
    return f"Tool response: {query}"

# Create tool from function
my_tool = Tool.from_function(
    func=my_tool_function,
    name="simple_tool",
    description="A simple tool"
)

# Tavily Search Tool
tavily_search = TavilySearchResults(max_results=2)

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Tools list
tools = [tavily_search, my_tool]

# Create agent
agent = initialize_agent(
    tools,
    llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent
response = agent.run("What's the weather like today in London?")

print(response)



> Entering new AgentExecutor chain...
I need to find the most current weather information for London. I'll use a search engine to get accurate and up-to-date results.  
Action: tavily_search_results_json  
Action Input: "current weather London"  
Observation: [{'url': 'https://en.climate-data.org/europe/united-kingdom-213/c/may-5', 'content': "| 28. May | 14 °C | 57 °F | 18 °C | 64 °F | 10 °C | 50 °F | 3.8 mm | 0.1 inch. |\n| 29. May | 14 °C | 57 °F | 18 °C | 65 °F | 9 °C | 49 °F | 5.3 mm | 0.2 inch. |\n| 30. May | 14 °C | 57 °F | 18 °C | 65 °F | 10 °C | 50 °F | 2.9 mm | 0.1 inch. |\n| 31. May | 15 °C | 58 °F | 19 °C | 66 °F | 10 °C | 50 °F | 1.8 mm | 0.1 inch. | [...] | 19. May | 10 °C | 50 °F | 14 °C | 56 °F | 6 °C | 44 °F | 0.0 mm | 0.0 inch. |\n| 20. May | 11 °C | 51 °F | 13 °C | 56 °F | 7 °C | 45 °F | 0.0 mm | 0.0 inch. |\n| 21. May | 11 °C | 51 °F | 14 °C | 57 °F | 7 °C | 45 °F | 0.0 mm | 0.0 inch. |\n| 22. May | 10 °C | 51 °F | 14 °C | 57 °F | 7 °C | 44 °F | 0.0 mm | 0.0 inch.

In [ ]:
prompt_template = "Summarize the following content: {content}"
llm = ChatOpenAI(model="gpt-4o-mini")

llm_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(prompt_template)
)

summarize_tool = Tool.from_function(
    func=llm_chain.run,
    name="Summarizer",
    description="Summarizes a web page"
)

In [ ]:
tools = [tavily_search, summarize_tool]

agent = initialize_agent(
    tools=tools,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm,
    verbose=True,handle_parsing_errors=True
)

In [ ]:
response = agent.invoke({"input": "Who invented the World Wide Web and what impact did it have?"})
print(response["output"])



> Entering new AgentExecutor chain...
I need to find information about the inventor of the World Wide Web and its impact. This involves searching for historical details and significance. 
Action: tavily_search_results_json
Action Input: "Who invented the World Wide Web and what impact did it have?"
Observation: [{'url': 'https://www.onemarketer.net/history-and-impact-of-the-internet', 'content': 'The true turning point came in1989, whenTim Berners-Leeinvented theWorld Wide Webat CERN. His vision of a publicly accessible hypertext system materialized in 1991 with the first website (info.cern.ch). The 1990s saw the birth of browsers likeMosaic (1993)and pioneering companies likeAmazon (1994)andGoogle (1998).\n\nThe new millennium broughtWeb 2.0, characterized by interactive platforms likeFacebook(2004)andYouTube (2005). Mobility became key with theiPhone (2007), shifting Internet access from computers to handheld devices. Today, the Internet is advancing towardartificial intelligence, 